In [1]:
# ==============================================================================
# ANALYSE DE LA SÉVÉRITÉ DES INCENDIES VIA SENTINEL-2 (SATELLITE COPERNICUS)
# Étude de cas : Incendie du Massif des Maures (Var, France) - Août 2021
# ==============================================================================

# ------------------------------------------------------------------------------
# ÉTAPE 1 — Importation des bibliothèques et configuration de l'environnement
# ------------------------------------------------------------------------------
import sys
import os
from pathlib import Path
import numpy as np
import rasterio
import rasterio.features
from rasterio.transform import Affine
import geopandas as gpd

# ------------------------------------------------------------------------------
# DÉFINITION ROBUSTE DU DOSSIER RACINE DU PROJET (PROJECT_DIR)
# ------------------------------------------------------------------------------
# On recherche automatiquement le dossier parent qui contient le répertoire 'src'
current_dir = Path.cwd().resolve()
PROJECT_DIR = None

# Remonte l'arborescence jusqu'à trouver le dossier contenant 'src'
for parent in [current_dir] + list(current_dir.parents):
    if (parent / "src").exists():
        PROJECT_DIR = parent
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        f"❌ Impossible de trouver le dossier racine du projet contenant 'src/'.\n"
        f"Dossier actuel : {current_dir}\n"
        f"Assurez-vous d'avoir ouvert VS Code dans le dossier du dépôt GitHub !"
    )

DATA_DIR = PROJECT_DIR / "data"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Ajout prioritaire du dossier racine dans sys.path pour importer 'src'
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Importation des fonctions métier personnalisées (Principe DRY : Don't Repeat Yourself)
from src.indices import nbr, nbr2, dnbr, ndvi, classify_severity, SEVERITY_LABELS
from src.visualize import SEVERITY_COLORS

print("✅ Environnement Python et modules 'src' chargés avec succès.")
print(f"   Racine du projet    : {PROJECT_DIR}")
print(f"   Version de Rasterio : {rasterio.__version__}")

✅ Environnement Python et modules 'src' chargés avec succès.
   Racine du projet    : /home/bronislaw/GitHub/sentinel2-burn-severity-analysis-python
   Version de Rasterio : 1.5.2


In [2]:
# ------------------------------------------------------------------------------
# ÉTAPE 2 — Définition des bandes spectrales et vérification des métadonnées
# ------------------------------------------------------------------------------
# Définition des chemins vers les bandes Sentinel-2 L2A :
# - Bande 04 (Rouge - 665 nm)     : Analyse de la chlorophylle (NDVI)
# - Bande 08 (PIR / NIR - 842 nm) : Sensible à la structure du feuillage
# - Bande 11 (SWIR1 - 1610 nm)    : Sensible au contenu en eau de la végétation/sol
# - Bande 12 (SWIR2 - 2190 nm)    : Sensible aux cendres, charbon et sol nu
# ------------------------------------------------------------------------------

PRE_FIRE_DIR = DATA_DIR / "pre_fire"
POST_FIRE_DIR = DATA_DIR / "post_fire"

# Chemins des bandes pré-incendie (28 Juillet 2021)
pre_b04_path = PRE_FIRE_DIR / "2021-07-28-00_00_2021-07-28-23_59_Sentinel-2_L2A_B04_(Raw).tiff"
pre_b08_path = PRE_FIRE_DIR / "2021-07-28-00_00_2021-07-28-23_59_Sentinel-2_L2A_B08_(Raw).tiff"
pre_b11_path = PRE_FIRE_DIR / "2021-07-28-00_00_2021-07-28-23_59_Sentinel-2_L2A_B11_(Raw).tiff"
pre_b12_path = PRE_FIRE_DIR / "2021-07-28-00_00_2021-07-28-23_59_Sentinel-2_L2A_B12_(Raw).tiff"

# Chemins des bandes post-incendie (27 Août 2021)
post_b04_path = POST_FIRE_DIR / "2021-08-27-00_00_2021-08-27-23_59_Sentinel-2_L2A_B04_(Raw).tiff"
post_b08_path = POST_FIRE_DIR / "2021-08-27-00_00_2021-08-27-23_59_Sentinel-2_L2A_B08_(Raw).tiff"
post_b11_path = POST_FIRE_DIR / "2021-08-27-00_00_2021-08-27-23_59_Sentinel-2_L2A_B11_(Raw).tiff"
post_b12_path = POST_FIRE_DIR / "2021-08-27-00_00_2021-08-27-23_59_Sentinel-2_L2A_B12_(Raw).tiff"

# Inspection rigoureuse des métadonnées géospatiales via Rasterio
with rasterio.open(pre_b08_path) as src:
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform
    width = src.width
    height = src.height
    dtype = src.dtypes[0]

    # CORRECTION CRITIQUE : Calcul dynamique de la résolution spatiale exacte
    # La matrice affine définit la taille du pixel : transform.a = largeur, transform.e = hauteur
    pixel_res_x = abs(transform.a)
    pixel_res_y = abs(transform.e)
    pixel_area_m2 = pixel_res_x * pixel_res_y

print("=" * 65)
print("  MÉTADONNÉES DE L'IMAGE SOURCE (SENTINEL-2)")
print("=" * 65)
print(f"  Système de projection (SCR) : {crs}")
print(f"  Dimensions de l'image       : {width} x {height} pixels")
print(f"  Résolution spatiale exacte  : {pixel_res_x:.2f} m x {pixel_res_y:.2f} m")
print(f"  Surface réelle d'un pixel   : {pixel_area_m2:.2f} m²")
print(f"  Type de données (Dtype)     : {dtype}")
print("=" * 65)

if dtype == 'uint8':
    print("\n⚠️  AVERTISSEMENT RADIOMÉTRIQUE :")
    print("    Les données sont codées en uint8 (8-bit, valeurs 0-255).")
    print("    Pour une précision spectrale maximale, privilégiez le format natif 16-bit (Reflectance L2A).")

  MÉTADONNÉES DE L'IMAGE SOURCE (SENTINEL-2)
  Système de projection (SCR) : EPSG:32632
  Dimensions de l'image       : 1185 x 1127 pixels
  Résolution spatiale exacte  : 7.29 m x 7.03 m
  Surface réelle d'un pixel   : 51.30 m²
  Type de données (Dtype)     : uint8

⚠️  AVERTISSEMENT RADIOMÉTRIQUE :
    Les données sont codées en uint8 (8-bit, valeurs 0-255).
    Pour une précision spectrale maximale, privilégiez le format natif 16-bit (Reflectance L2A).


In [3]:
# ------------------------------------------------------------------------------
# ÉTAPE 3 — Ingestion des bandes et calcul du dNBR (Normalized Burn Ratio)
# ------------------------------------------------------------------------------
# Principe physique de l'indice NBR :
# NBR = (NIR - SWIR2) / (NIR + SWIR2)
#
# - Végétation saine  : Réflectance forte dans le PIR/NIR (B08) et faible dans le SWIR2 (B12).
# - Végétation brûlée : Destruction de la structure cellulaire (chute du PIR) et 
#                       apparition de sol nu/cendres (forte hausse du SWIR2).
#
# dNBR = NBR_pre - NBR_post
# Plus la valeur de dNBR est élevée, plus la dégradation du couvert végétal est sévère.
# ------------------------------------------------------------------------------

def load_raster_band(filepath: Path) -> np.ndarray:
    """
    Charge une bande GeoTIFF sous forme de tableau multidimensionnel NumPy (2D array).
    
    Conversion en float32 :
    Indispensable pour autoriser les valeurs décimales (ex: -1.0 à +1.0) et éviter 
    les erreurs d'arrondi ou de saturation propres au format uint8 (0 à 255).
    """
    with rasterio.open(filepath) as src:
        return src.read(1).astype(np.float32)

# Chargement des bandes Proche Infrarouge (NIR - B08) et Infrarouge Moyen (SWIR2 - B12)
pre_nir   = load_raster_band(pre_b08_path)
pre_swir  = load_raster_band(pre_b12_path)
post_nir  = load_raster_band(post_b08_path)
post_swir = load_raster_band(post_b12_path)

# Vérification d'intégrité spatiale : toutes les matrices doivent partager les mêmes dimensions
assert pre_nir.shape == pre_swir.shape == post_nir.shape == post_swir.shape, \
    "❌ Erreur : Les dimensions des bandes ne concordent pas !"

# Calcul du NBR pré- et post-incendie via notre module local 'src.indices' (Principe DRY)
nbr_pre  = nbr(pre_nir, pre_swir)
nbr_post = nbr(post_nir, post_swir)

# Calcul du dNBR (delta NBR)
dnbr_array = dnbr(nbr_pre, nbr_post)

print("✅ Calculs matriciels NBR et dNBR exécutés avec succès.")
print(f"   Forme du tableau NumPy (Hauteur x Largeur) : {dnbr_array.shape}")
print(f"   Plage spectrale dNBR                       : {dnbr_array.min():.3f} à {dnbr_array.max():.3f}")

✅ Calculs matriciels NBR et dNBR exécutés avec succès.
   Forme du tableau NumPy (Hauteur x Largeur) : (1127, 1185)
   Plage spectrale dNBR                       : -0.800 à 1.343


In [4]:
# ------------------------------------------------------------------------------
# ÉTAPE 4 — Classification de la sévérité (USGS) et statistiques de surfaces
# ------------------------------------------------------------------------------
# Légende des seuils USGS pour l'indice dNBR :
# Classe 0 (Unburned)  : dNBR < 0.10     -> Zone non brûlée ou régénération
# Classe 1 (Low)       : 0.10 <= dNBR < 0.27 -> Sévérité faible (litière touchée)
# Classe 2 (Moderate)  : 0.27 <= dNBR < 0.44 -> Sévérité modérée (arbres partiellement touchés)
# Classe 3 (High)      : 0.44 <= dNBR < 0.66 -> Sévérité haute (canopée brûlée)
# Classe 4 (Very High) : dNBR >= 0.66    -> Sévérité très haute (destruction totale)
# ------------------------------------------------------------------------------

# Reclassification matricielle via le module 'src.indices'
severity_array = classify_severity(dnbr_array)

# Comptage des pixels par classe de sévérité
unique_classes, pixel_counts = np.unique(severity_array, return_counts=True)
total_pixels = severity_array.size

print("=" * 70)
print("  BILAN STATISTIQUE DES SURFACES BRÛLÉES (MASSIF DES MAURES)")
print("=" * 70)
print(f"{'Classe de Sévérité':<20} {'Pixels':>10} {'Surface (ha)':>15} {'Surface (km²)':>15} {'Proportion (%)':>12}")
print("-" * 70)

severity_stats = {}
for cls_id, count in zip(unique_classes, pixel_counts):
    label = SEVERITY_LABELS[cls_id]
    
    # CALCUL CORRIGÉ : Prise en compte de la surface exacte calculée à l'Étape 2 (51.30 m²)
    area_ha  = (count * pixel_area_m2) / 10_000.0   # 1 ha = 10 000 m²
    area_km2 = (count * pixel_area_m2) / 1_000_000.0 # 1 km² = 1 000 000 m²
    pct      = (count / total_pixels) * 100.0
    
    severity_stats[label] = {
        'pixels': int(count),
        'ha': area_ha,
        'km2': area_km2,
        'pct': pct
    }
    
    print(f"  {label:<18} {count:>10,} {area_ha:>15.1f} {area_km2:>15.2f} {pct:>11.1f} %")

# Surface totale affectée par le feu (classes 1 à 4 excluant 'Unburned')
total_burned_ha = sum(data['ha'] for label, data in severity_stats.items() if label != 'Unburned')
total_burned_km2 = sum(data['km2'] for label, data in severity_stats.items() if label != 'Unburned')

print("-" * 70)
print(f"  {'SURFACE TOTALE BRÛLÉE':<20} {'':>10} {total_burned_ha:>15.1f} ha {total_burned_km2:>15.2f} km²")
print("=" * 70)

  BILAN STATISTIQUE DES SURFACES BRÛLÉES (MASSIF DES MAURES)
Classe de Sévérité       Pixels    Surface (ha)   Surface (km²) Proportion (%)
----------------------------------------------------------------------
  Unburned              887,735          4554.2           45.54        66.5 %
  Low                    85,504           438.6            4.39         6.4 %
  Moderate               56,021           287.4            2.87         4.2 %
  High                   65,010           333.5            3.34         4.9 %
  Very High             241,225          1237.5           12.38        18.1 %
----------------------------------------------------------------------
  SURFACE TOTALE BRÛLÉE                     2297.1 ha           22.97 km²


In [5]:
# ------------------------------------------------------------------------------
# ÉTAPE 5 — Exportation des rasters géoréférencés (Format GeoTIFF)
# ------------------------------------------------------------------------------
# Nous exportons 3 produits rasters dans le dossier 'outputs/' :
# 1. dnbr.tif        : Valeurs continues décimales de dNBR (float32)
# 2. severity.tif    : Classes de sévérité de 0 à 4 (uint8)
# 3. severity_rgb.tif: Carte couleur RVB prête à l'affichage (3 bandes uint8)
# ------------------------------------------------------------------------------

# Table de couleurs officielle pour la cartographie de la sévérité
color_map = {
    0: (34, 139, 34),    # Non brûlé (Unburned)  -> Vert forêt
    1: (255, 255, 0),    # Faible (Low)          -> Jaune
    2: (255, 165, 0),    # Modérée (Moderate)    -> Orange
    3: (255, 0, 0),      # Haute (High)          -> Rouge
    4: (139, 0, 0),      # Très Haute (Very High)-> Rouge foncé / Marron
}

# Mise à jour du profil géospatial avec compression LZW (Optimisation du stockage)
export_profile = profile.copy()
export_profile.update(
    compress='lzw',
    nodata=None
)

# 1) Export du dNBR en float32
export_profile.update(dtype=rasterio.float32, count=1)
with rasterio.open(OUTPUTS_DIR / 'dnbr.tif', 'w', **export_profile) as dst:
    dst.write(dnbr_array.astype(np.float32), 1)

# 2) Export de la sévérité reclassée en uint8
export_profile.update(dtype=rasterio.uint8, count=1)
with rasterio.open(OUTPUTS_DIR / 'severity.tif', 'w', **export_profile) as dst:
    dst.write(severity_array.astype(np.uint8), 1)

# 3) Export de la carte de sévérité RVB (3 bandes : Rouge, Vert, Bleu)
h, w = severity_array.shape
rgb_array = np.zeros((3, h, w), dtype=np.uint8)
for class_val, color in color_map.items():
    mask = (severity_array == class_val)
    for band_idx, channel_color in enumerate(color):
        rgb_array[band_idx][mask] = channel_color

export_profile.update(dtype=rasterio.uint8, count=3)
with rasterio.open(OUTPUTS_DIR / 'severity_rgb.tif', 'w', **export_profile) as dst:
    dst.write(rgb_array)

print("✅ Fichiers GeoTIFFs exportés avec succès dans 'outputs/' :")
print("   - dnbr.tif (float32)")
print("   - severity.tif (uint8)")
print("   - severity_rgb.tif (uint8 RVB 3 bandes)")

✅ Fichiers GeoTIFFs exportés avec succès dans 'outputs/' :
   - dnbr.tif (float32)
   - severity.tif (uint8)
   - severity_rgb.tif (uint8 RVB 3 bandes)


In [6]:
# ------------------------------------------------------------------------------
# ÉTAPE 6 — Calcul des indices secondaires (dNDVI et dNBR2)
# ------------------------------------------------------------------------------
# 1. dNDVI (delta NDVI) : Mesure la perte pure de chlorophylle/vigueur végétale.
#    NDVI = (NIR - RED) / (NIR + RED)  -> Utilise la bande B04 (Rouge)
#
# 2. dNBR2 (delta NBR2) : Utilise les deux bandes Infrarouge Moyen (B11 et B12).
#    NBR2 = (SWIR1 - SWIR2) / (SWIR1 + SWIR2)
#    Très sensible à l'humidité du sol et au dépôt de carbone/charbon.
# ------------------------------------------------------------------------------

# Chargement des bandes Rouge (B04) et SWIR1 (B11)
pre_red    = load_raster_band(pre_b04_path)
post_red   = load_raster_band(post_b04_path)
pre_swir1  = load_raster_band(pre_b11_path)
post_swir1 = load_raster_band(post_b11_path)

# Calcul du dNDVI
ndvi_pre  = ndvi(pre_nir, pre_red)
ndvi_post = ndvi(post_nir, post_red)
dndvi_array = ndvi_pre - ndvi_post

# Calcul du dNBR2
nbr2_pre   = nbr2(pre_swir1, pre_swir)
nbr2_post  = nbr2(post_swir1, post_swir)
dnbr2_array = nbr2_pre - nbr2_post

# Exportation des rasters dNDVI et dNBR2
export_profile.update(dtype=rasterio.float32, count=1)

with rasterio.open(OUTPUTS_DIR / 'dndvi.tif', 'w', **export_profile) as dst:
    dst.write(dndvi_array.astype(np.float32), 1)

with rasterio.open(OUTPUTS_DIR / 'dnbr2.tif', 'w', **export_profile) as dst:
    dst.write(dnbr2_array.astype(np.float32), 1)

print("✅ Indices secondaires calculés et exportés :")
print(f"   Plage dNDVI : {dndvi_array.min():.3f} à {dndvi_array.max():.3f} -> outputs/dndvi.tif")
print(f"   Plage dNBR2 : {dnbr2_array.min():.3f} à {dnbr2_array.max():.3f} -> outputs/dnbr2.tif")

✅ Indices secondaires calculés et exportés :
   Plage dNDVI : -1.103 à 0.836 -> outputs/dndvi.tif
   Plage dNBR2 : -0.333 à 0.571 -> outputs/dnbr2.tif


In [7]:
# ------------------------------------------------------------------------------
# ÉTAPE 7 — Vectorisation géospatiale des périmètres (Format GeoJSON) - CORRIGÉE
# ------------------------------------------------------------------------------
from rasterio.features import shapes, sieve
from shapely.geometry import shape

# 1) Filtrage du bruit spatial : supprime les groupes de pixels isolés moin de 10 pixels (~500 m²)
severity_cleaned = sieve(severity_array.astype(np.uint8), size=10)

# 2) Extraction des formes géométriques (polygones) pour les zones brûlées (classes 1 à 4)
mask_burned = (severity_cleaned > 0)

shapes_generator = shapes(
    severity_cleaned,
    mask=mask_burned,
    transform=transform  # Matrice affine d'origine
)

# Reconstruction des géométries Shapely avec attributs À PLAT
records = []
for geom, val in shapes_generator:
    val_int = int(val)
    records.append({
        'geometry': shape(geom),
        'severity_code': val_int,
        'severity_label': SEVERITY_LABELS[val_int]
    })

# Création du GeoDataFrame dans le système de projection métrique natif (UTM Zone 32N)
gdf = gpd.GeoDataFrame.from_records(records)
gdf = gdf.set_crs(crs)

# 3) Calcul EXACT de la surface de chaque polygone en hectares (1 ha = 10 000 m²)
gdf['area_ha'] = gdf.geometry.area / 10_000.0

# 4) Simplification géométrique modérée (10 mètres) pour alléger le fichier GeoJSON
gdf['geometry'] = gdf.simplify(tolerance=10.0, preserve_topology=True)

# 5) Reprojection finale vers EPSG:4326 (WGS84) pour l'export GeoJSON
gdf_wgs84 = gdf.to_crs(epsg=4326)

# Exportation vers le dossier 'outputs/'
geojson_path = OUTPUTS_DIR / 'burn_severity.geojson'
gdf_wgs84.to_file(geojson_path, driver='GeoJSON')

print("✅ Vectorisation géospatiale réussie et corrigée :")
print(f"   - Colonnes du GeoDataFrame : {list(gdf.columns)}")
print(f"   - Nombre de polygones extraits : {len(gdf_wgs84):,}")
print(f"   - Surface totale vectorisée   : {gdf['area_ha'].sum():.1f} ha")

✅ Vectorisation géospatiale réussie et corrigée :
   - Colonnes du GeoDataFrame : ['geometry', 'severity_code', 'severity_label', 'area_ha']
   - Nombre de polygones extraits : 2,141
   - Surface totale vectorisée   : 2289.2 ha


In [8]:
# ------------------------------------------------------------------------------
# ÉTAPE 8 — Rendu cartographique composite 2x2 avec titres (PIL)
# ------------------------------------------------------------------------------
# Génère une figure comparative de haute qualité (2400x2300 px) regroupant 
# les 4 indices spectraux avec leurs titres respectifs centrés.
# ------------------------------------------------------------------------------

from PIL import Image, ImageDraw, ImageFont

def array_to_rgb_visual(arr: np.ndarray, vmin: float, vmax: float, mode: str = 'fire') -> np.ndarray:
    """Convertit une matrice 2D en tableau RVB 8-bit avec étalonnage du contraste."""
    norm = np.clip((arr - vmin) / (vmax - vmin + 1e-10), 0, 1)
    if mode == 'ndvi':
        norm = 1.0 - norm
    
    h, w = norm.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    rgb[..., 0] = (norm * 255).astype(np.uint8)
    rgb[..., 1] = ((1 - np.abs(norm - 0.5) * 2) * 255).astype(np.uint8)
    return rgb

# 1) Détection multiplateforme des polices de caractères (Linux / Windows / Fallback)
font = None
font_paths = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",  # Linux Ubuntu / Debian
    "/usr/share/fonts/TTF/DejaVuSans-Bold.ttf",              # Arch Linux
    "C:/Windows/Fonts/arialbd.ttf",                          # Windows Bold
    "C:/Windows/Fonts/arial.ttf",                            # Windows Standard
]

for fpath in font_paths:
    if os.path.exists(fpath):
        try:
            font = ImageFont.truetype(fpath, 28)
            break
        except Exception:
            pass

if font is None:
    # Police par défaut si aucune police système n'est trouvée
    font = ImageFont.load_default()

# 2) Conversion des matrices en rendus visuels RVB
img_dnbr  = array_to_rgb_visual(dnbr_array,  vmin=-0.2, vmax=0.8, mode='fire')
img_dnbr2 = array_to_rgb_visual(dnbr2_array, vmin=-0.2, vmax=0.5, mode='fire')
img_dndvi = array_to_rgb_visual(dndvi_array, vmin=-0.2, vmax=0.6, mode='ndvi')

# Composition de la carte de sévérité basée sur la table de couleurs
h, w = severity_array.shape
img_sev = np.zeros((h, w, 3), dtype=np.uint8)
for val, color in color_map.items():
    img_sev[severity_array == val] = color

# 3) Assemblage du panneau composite 2x2 avec marges et en-têtes
GAP = 40
PADDING_TOP = 60
total_w = w * 2 + GAP
total_h = h * 2 + GAP + PADDING_TOP
canvas = np.full((total_h, total_w, 3), 255, dtype=np.uint8) # Fond blanc

grid = [
    (img_dnbr,  '1. dNBR (Indice Principal B08 - B12)', 0, 0),
    (img_sev,   '2. Sévérité de l\'Incendie (USGS)',    1, 0),
    (img_dndvi, '3. dNDVI (Perte de Chlorophylle)',    0, 1),
    (img_dnbr2, '4. dNBR2 (Réponse SWIR1 - SWIR2)',     1, 1),
]

# Placement des 4 images sur le canevas
for panel, _, col, row in grid:
    x_off = col * (w + GAP)
    y_off = PADDING_TOP + row * (h + GAP)
    canvas[y_off:y_off + h, x_off:x_off + w] = panel

result_img = Image.fromarray(canvas)
draw = ImageDraw.Draw(result_img)

# Dessin des titres centrés au-dessus de chaque carte
for _, title, col, row in grid:
    x_off = col * (w + GAP)
    y_off = PADDING_TOP + row * (h + GAP) - 40
    
    # Calcul de la largeur du texte pour un centrage parfait
    bbox = draw.textbbox((0, 0), title, font=font)
    text_w = bbox[2] - bbox[0]
    x_centered = x_off + (w - text_w) // 2
    
    draw.text((x_centered, max(y_off, 10)), title, fill=(0, 0, 0), font=font)

# Sauvegarde des images finales dans outputs/
overview_path = OUTPUTS_DIR / 'overview.png'
severity_png_path = OUTPUTS_DIR / 'burn_severity.png'

result_img.save(overview_path)
Image.fromarray(img_sev).save(severity_png_path)

print(f"✅ Carte composite d'ensemble générée avec succès ({total_w} x {total_h} px) :")
print(f"   - outputs/overview.png")
print(f"   - outputs/burn_severity.png")

✅ Carte composite d'ensemble générée avec succès (2410 x 2354 px) :
   - outputs/overview.png
   - outputs/burn_severity.png


In [9]:
# ------------------------------------------------------------------------------
# ÉTAPE 9 — Bilan synthétique et statistiques finales (Tableau Pandas)
# ------------------------------------------------------------------------------
# Cette dernière étape agrège l'ensemble des résultats (Rasters et Wecteurs)
# sous la forme d'un tableau récapitulatif propre (DataFrame Pandas).
# ------------------------------------------------------------------------------

import pandas as pd

# 1) Calcul des surfaces vectorielles par classe à partir du GeoDataFrame (EPSG:32632)
vector_areas = gdf.groupby('severity_code')['area_ha'].sum().to_dict()

# 2) Construction du tableau récapitulatif
summary_data = []

for cls_id, label in enumerate(SEVERITY_LABELS):
    raster_ha = severity_stats[label]['ha']
    vector_ha = vector_areas.get(cls_id, 0.0)
    pct = severity_stats[label]['pct']
    
    summary_data.append({
        'Code USGS': cls_id,
        'Classe de Sévérité': label,
        'Surface Raster (ha)': round(raster_ha, 1),
        'Surface Vectorielle (ha)': round(vector_ha, 1),
        'Proportion (%)': round(pct, 1)
    })

# Création du DataFrame Pandas
df_summary = pd.DataFrame(summary_data)

# Calcul de la ligne de Total pour les zones brûlées (Classes 1 à 4)
burned_df = df_summary[df_summary['Code USGS'] > 0]
total_raster_burned = burned_df['Surface Raster (ha)'].sum()
total_vector_burned = burned_df['Surface Vectorielle (ha)'].sum()
total_pct_burned = burned_df['Proportion (%)'].sum()

# Ajout d'une ligne de total au DataFrame
df_summary.loc[len(df_summary)] = {
    'Code USGS': '-',
    'Classe de Sévérité': 'TOTAL BRÛLÉ (Classes 1-4)',
    'Surface Raster (ha)': round(total_raster_burned, 1),
    'Surface Vectorielle (ha)': round(total_vector_burned, 1),
    'Proportion (%)': round(total_pct_burned, 1)
}

print("=" * 80)
print("  BILAN DÉFINITIF DE L'INCENDIE DU MASSIF DES MAURES (AOÛT 2021)")
print("=" * 80)

# Affichage propre du tableau Pandas
display(df_summary) # En notebook Jupyter, display() affiche un beau tableau HTML

print("\n📌 CONCLUSIONS ANALYTIQUES :")
print(f"   1. L'emprise étudiée couvre une superficie totale de {df_summary.iloc[:-1]['Surface Raster (ha)'].sum():,.1f} ha.")
print(f"   2. La surface totale affectée par le feu s'élève à {total_raster_burned:,.1f} ha (soit {total_pct_burned:.1f}% de la zone).")
print(f"   3. Les classes de haute et très haute sévérité (red/darkred) représentent la majorité des dégâts.")
print(f"   4. L'écart entre la surface raster ({total_raster_burned:.1f} ha) et vectorielle ({total_vector_burned:.1f} ha)")
print(f"      s'explique par le filtrage spatial ('sieve') et la simplification géométrique de 10 m.")
print("=" * 80)

  BILAN DÉFINITIF DE L'INCENDIE DU MASSIF DES MAURES (AOÛT 2021)


/tmp/ipykernel_17007/1133525748.py:39: Pandas4Warning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. The existing dtype int64 cannot hold the value being set, so the result has dtype object. Cast the object to object before this operation to retain the current behavior.
  df_summary.loc[len(df_summary)] = {


,Code USGS,Classe de Sévérité,Surface Raster (ha),Surface Vectorielle (ha),Proportion (%)
0,0,Unburned,4554.2,0.0,66.5
1,1,Low,438.6,452.2,6.4
2,2,Moderate,287.4,263.4,4.2
3,3,High,333.5,319.3,4.9
4,4,Very High,1237.5,1254.4,18.1
5,-,TOTAL BRÛLÉ (Classes 1-4),2297.0,2289.3,33.6



📌 CONCLUSIONS ANALYTIQUES :
   1. L'emprise étudiée couvre une superficie totale de 6,851.2 ha.
   2. La surface totale affectée par le feu s'élève à 2,297.0 ha (soit 33.6% de la zone).
   3. Les classes de haute et très haute sévérité (red/darkred) représentent la majorité des dégâts.
   4. L'écart entre la surface raster (2297.0 ha) et vectorielle (2289.3 ha)
      s'explique par le filtrage spatial ('sieve') et la simplification géométrique de 10 m.
